# Notebook 02b: Category Classification (Problem B)

Multi-label incident category classification using a three-tier approach:
- **Tier 1 (Baseline):** TF-IDF + Logistic Regression (OVR)
- **Tier 2 (Ablation):** Sentence-BERT embeddings + Logistic Regression
- **Tier 3 (Fusion):** Text Tower + Tabular features → MLP per-label heads

Taxonomy v2 (calibrated against real ASRS keyword values):

| Category | Source Keywords |
|----------|-----------------|
| Flight_Operations | Human Factors, Procedure, Company Policy |
| Equipment_System | Aircraft, Equipment/Tooling, MEL, Software |
| ATC_Communication | ATC Equipment, Airspace Structure, Airport |
| Environment | Weather, Environment - Non Weather Related |
| Airspace_Navigation | Chart Or Publication |

In [ ]:
import sys
from pathlib import Path

# Resolve project root regardless of CWD
ROOT = Path.cwd()
while not (ROOT / 'pyproject.toml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f'Project root: {ROOT}')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.config import set_seeds, resolve_path
from src.data.loader import load_raw_data, parse_time_date, load_main_config
from src.data.target_engineering import apply_severity_rubric, apply_category_taxonomy
from src.features.temporal import create_temporal_split
from src.features.text import preprocess_narratives, combine_text_fields
from src.evaluation.multilabel_metrics import multilabel_report

set_seeds()
config = load_main_config()
print('Setup complete.')

## 1. Load and Prepare Data

In [ ]:
df = load_raw_data(config)
df = apply_severity_rubric(df)
df = parse_time_date(df)
df = create_temporal_split(df)
df, cat_matrix = apply_category_taxonomy(df)
df = preprocess_narratives(df)
df = combine_text_fields(df, output_col='combined_text')

label_names = sorted(cat_matrix.columns.tolist())
print(f'Dataset: {len(df)} rows')
print(f'Labels: {label_names}')
print(f'\nLabel counts (all data):')
display(cat_matrix.sum().rename('count').to_frame())

In [ ]:
# Per-split label distribution
split_stats = []
for sp in ['train', 'val', 'test']:
    mask = df['split'] == sp
    for label in label_names:
        n = cat_matrix.loc[mask, label].sum()
        pct = n / mask.sum() * 100
        split_stats.append({'split': sp, 'label': label, 'count': n, 'pct': pct})
stats_df = pd.DataFrame(split_stats)

fig, ax = plt.subplots(figsize=(12, 5))
for i, label in enumerate(label_names):
    sub = stats_df[stats_df['label'] == label]
    ax.bar([j + i * 0.15 for j in range(3)], sub['pct'], width=0.14, label=label)
ax.set_xticks([0.3, 1.3, 2.3])
ax.set_xticklabels(['Train', 'Val', 'Test'])
ax.set_ylabel('Prevalence (%)')
ax.set_title('Multi-label Category Prevalence by Split')
ax.legend(bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()

## 2. Tier 1 — TF-IDF Baseline

In [ ]:
from src.models.category import build_tfidf_baseline, predict_tfidf

tr_mask = df['split'] == 'train'
va_mask = df['split'] == 'val'
te_mask = df['split'] == 'test'

tier1_model = build_tfidf_baseline(
    df.loc[tr_mask, 'combined_text'], cat_matrix.loc[tr_mask],
    df.loc[va_mask, 'combined_text'], cat_matrix.loc[va_mask],
    label_names,
)
print('Thresholds:', tier1_model['thresholds'])

In [ ]:
_, t1_preds = predict_tfidf(tier1_model, df.loc[te_mask, 'combined_text'])
t1_report = multilabel_report(cat_matrix.loc[te_mask].values, t1_preds, label_names)

print('=== Tier 1 (TF-IDF) Test Results ===')
print(f"  Macro-F1:   {t1_report['macro_f1']:.4f}")
print(f"  Micro-F1:   {t1_report['micro_f1']:.4f}")
print(f"  Hamming:    {t1_report['hamming_loss']:.4f}")
print(f"  Subset Acc: {t1_report['subset_accuracy']:.4f}")
print('\nPer-label F1:')
for label in label_names:
    f1 = t1_report['per_label'][label]['f1']
    print(f'  {label:<28} {f1:.3f}')

## 3. Tier 2 — Sentence-BERT Text Tower

In [ ]:
from src.models.category import encode_texts, build_text_tower, predict_text_tower

emb_path = resolve_path('data/processed')

# Load cached embeddings if available, otherwise encode
if (emb_path / 'emb_train.npy').exists():
    print('Loading cached SBERT embeddings...')
    emb_train = np.load(emb_path / 'emb_train.npy')
    emb_val   = np.load(emb_path / 'emb_val.npy')
    emb_test  = np.load(emb_path / 'emb_test.npy')
    print(f'Embedding dim: {emb_train.shape[1]}')
else:
    print('Encoding texts with SBERT (this takes ~15-30min on CPU)...')
    emb_train = encode_texts(df.loc[tr_mask, 'combined_text'], batch_size=128)
    emb_val   = encode_texts(df.loc[va_mask, 'combined_text'], batch_size=128)
    emb_test  = encode_texts(df.loc[te_mask, 'combined_text'], batch_size=128)
    np.save(emb_path / 'emb_train.npy', emb_train)
    np.save(emb_path / 'emb_val.npy',   emb_val)
    np.save(emb_path / 'emb_test.npy',  emb_test)
    print('Embeddings saved.')

In [ ]:
tier2_model = build_text_tower(
    emb_train, cat_matrix.loc[tr_mask],
    emb_val,   cat_matrix.loc[va_mask],
    label_names,
)
_, t2_preds = predict_text_tower(tier2_model, emb_test)
t2_report = multilabel_report(cat_matrix.loc[te_mask].values, t2_preds, label_names)

print('=== Tier 2 (SBERT Text Tower) Test Results ===')
print(f"  Macro-F1:   {t2_report['macro_f1']:.4f}")
print(f"  Micro-F1:   {t2_report['micro_f1']:.4f}")
print(f"  Hamming:    {t2_report['hamming_loss']:.4f}")
for label in label_names:
    f1 = t2_report['per_label'][label]['f1']
    print(f'  {label:<28} {f1:.3f}')

## 4. Tier 3 — Fusion Model

In [ ]:
# Load fusion model (trained by run_phase3.py --tier 3)
# Or run from the script: python scripts/run_phase3.py --tier 3
from src.models.category import load_category_model
fusion_path = resolve_path('models/category_fusion.joblib')
if fusion_path.exists():
    fusion = load_category_model(fusion_path)
    print('Loaded fusion model.')
else:
    print('Fusion model not yet trained. Run: python scripts/run_phase3.py --tier 3')

## 5. Ablation Comparison

In [ ]:
results = {
    'TF-IDF Baseline': t1_report,
    'SBERT Text Tower': t2_report,
}

metrics = ['macro_f1', 'micro_f1', 'hamming_loss', 'subset_accuracy']
summary = pd.DataFrame({
    name: {m: r[m] for m in metrics}
    for name, r in results.items()
}).T
display(summary.round(4))

# Per-label comparison chart
fig, ax = plt.subplots(figsize=(12, 5))
x = range(len(label_names))
width = 0.35
for i, (name, r) in enumerate(results.items()):
    f1s = [r['per_label'][l]['f1'] for l in label_names]
    ax.bar([xi + i * width for xi in x], f1s, width=width, label=name, alpha=0.8)
ax.set_xticks([xi + width/2 for xi in x])
ax.set_xticklabels(label_names, rotation=20, ha='right')
ax.set_ylabel('F1 Score')
ax.set_title('Per-Label F1: TF-IDF vs SBERT Text Tower')
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()